In [1]:
import nest_asyncio
nest_asyncio.apply()  # Cho phép nest event loop trong Jupyter

import sys
sys.path.insert(0, "/home/duoc/workflow/api")  # thư mục chứa folder app/
print("Cấu hình sys.path thành công! Thư mục hiện tại:")
import os
os.chdir("/home/duoc/workflow/api")  # Thêm dòng này TRƯỚC khi import app

Cấu hình sys.path thành công! Thư mục hiện tại:


1. Người dùng muốn gì?

→ Node 1

2. Thị trường đang viết gì để rank?

→ Node 2

3. Mình phải viết gì để thắng?

→ Node 3

In [2]:
import asyncio
from urllib.parse import quote
from bs4 import BeautifulSoup
import nodriver as uc

QUERY = "nhà hàng quán nhậu hải sản Đà Nẵng"


# =========================
# CLEAN
# =========================
def clean(text: str):
    if not text:
        return None

    text = " ".join(text.split())

    BAD = [
        "facebook",
        "tripadvisor",
        "đăng nhập",
        "chính sách",
        "hỗ trợ",
        "video",
        "youtube",
        "maps",
        "translate",
        "google",
        "javascript",
        "cookie"
    ]

    if (
        len(text) < 5
        or len(text) > 160
        or any(x in text.lower() for x in BAD)
    ):
        return None

    return text


def is_valid_url(url: str):
    if not url or not url.startswith("http"):
        return False

    BAD_URL = [
        "google.com/search",
        "googleusercontent",
        "webcache",
        "/settings",
        "accounts.google"
    ]

    return not any(x in url for x in BAD_URL)


# =========================
# NODE 1 ENGINE
# =========================
async def node1_serp():

    browser = await uc.start(headless=False)

    try:
        page = await browser.get(
            f"https://www.google.com/search?q={quote(QUERY)}&hl=vi"
        )

        await asyncio.sleep(3)

        soup = BeautifulSoup(await page.get_content(), "html.parser")

        # =========================
        # 1. TOP URLS (Organic)
        # =========================
        top_urls = []
        seen = set()

        for a in soup.select("a"):
            h3 = a.select_one("h3")
            if not h3:
                continue

            url = a.get("href")
            title = clean(h3.get_text(" ", strip=True))

            if not title or not is_valid_url(url):
                continue

            key = url.split("#")[0]
            if key in seen:
                continue

            seen.add(key)

            domain = url.split("/")[2] if "://" in url else ""

            top_urls.append({
                "position": len(top_urls) + 1,
                "title": title,
                "url": url,
                "domain": domain,
                "type": "organic"
            })

        # =========================
        # 2. PEOPLE ALSO ASK (PAA)
        # =========================
        paa = []

        for el in soup.select('div[jsname="N760b"]'):
            q = clean(el.get_text(" ", strip=True))
            if q and "?" in q:
                paa.append({
                    "question": q,
                    "intent": "expansion"
                })

        paa = paa[:5]

        # =========================
        # 3. RELATED SEARCHES
        # =========================
        related = []

        for a in soup.select("a"):
            text = clean(a.get_text(" ", strip=True))

            if not text:
                continue

            if any(k in text.lower() for k in ["quán", "hải sản", "đà nẵng"]):
                if len(text.split()) <= 10:
                    related.append(text)

        related = list(dict.fromkeys(related))[:10]

        # =========================
        # 4. SNIPPETS (clean SERP preview)
        # =========================
        snippets = []

        for block in soup.select("div.g"):
            snippet_el = block.select_one("span")

            if not snippet_el:
                continue

            text = clean(snippet_el.get_text(" ", strip=True))

            if text and 60 < len(text) < 300:
                snippets.append({
                    "snippet": text[:200]
                })

        snippets = snippets[:5]

        # =========================
        # 5. KEYWORD CLUSTER
        # =========================
        keyword_cluster = [
            "quán hải sản ngon Đà Nẵng",
            "hải sản giá rẻ",
            "gần biển Mỹ Khê",
            "Sơn Trà",
            "quán nhậu Đà Nẵng",
            "buffet hải sản"
        ]

        # =========================
        # 6. CONTENT ANGLE
        # =========================
        content_angle = [
            "review",
            "top list",
            "trải nghiệm",
            "giá rẻ",
            "du lịch ăn uống"
        ]

        # =========================
        # 7. INTENT LAYER
        # =========================
        intent = [
            "ăn nhậu",
            "du lịch",
            "gia đình",
            "buffet",
            "hải sản tươi sống"
        ]

        # =========================
        # 8. COMPETITOR PATTERN
        # =========================
        competitor_pattern = [
            "Pasgo",
            "Tripadvisor",
            "Blog du lịch SEO",
            "Facebook group",
            "Local listing / directory"
        ]

        # =========================
        # FINAL OUTPUT
        # =========================
        return {
            "query": QUERY,
            "top_urls": top_urls[:10],
            "people_also_ask": paa,
            "related_searches": related,
            "snippets": snippets,
            "keyword_cluster": keyword_cluster,
            "content_angle": content_angle,
            "intent": intent,
            "competitor_pattern": competitor_pattern
        }

    finally:
        if browser:
            browser.stop()


# =========================
# JUPYTER RUN
# =========================
node1 = await node1_serp()
from pprint import pprint
pprint(node1)

{'competitor_pattern': ['Pasgo',
                        'Tripadvisor',
                        'Blog du lịch SEO',
                        'Facebook group',
                        'Local listing / directory'],
 'content_angle': ['review',
                   'top list',
                   'trải nghiệm',
                   'giá rẻ',
                   'du lịch ăn uống'],
 'intent': ['ăn nhậu', 'du lịch', 'gia đình', 'buffet', 'hải sản tươi sống'],
 'keyword_cluster': ['quán hải sản ngon Đà Nẵng',
                     'hải sản giá rẻ',
                     'gần biển Mỹ Khê',
                     'Sơn Trà',
                     'quán nhậu Đà Nẵng',
                     'buffet hải sản'],
 'people_also_ask': [],
 'query': 'nhà hàng quán nhậu hải sản Đà Nẵng',
 'related_searches': ['Hải sản Thời Cổ',
                      'Hải sản Năm Đảnh',
                      'Hải sản Mộc quán',
                      'Hải Sản Gành Bàng',
                      'Quán Eo Biển',
                      'Hải 

In [3]:
import json
import re
from groq import Groq
from app.llm_clients import call_groq


from app.llm_clients import call_groq


def node2_llm(node1):

    payload = {
        "query": node1.get("query", ""),
        "top_urls": [x.get("title") for x in node1.get("top_urls", [])],
        "related_searches": node1.get("related_searches", []),
        "snippets": node1.get("snippets", [])
    }

    prompt = f"""
You are a SERP understanding layer.

You MUST output ONLY in this structure:

INTENT:
- ...

EXPANSION:
- ...

CONTENT SIGNALS:
- price:
- quality:
- format:
- location:

SERP TYPE:
- one or more labels

ENTITIES:
- only real named entities from input

MARKET INSIGHT:
- one sentence only

CRITICAL RULES:
- DO NOT repeat this instruction
- DO NOT output RULES section
- DO NOT add any extra headings
- ONLY use the sections above

INPUT:
{payload}
"""

    return call_groq(prompt)

# =========================
# 3. RUN
# =========================
node2 = node2_llm(node1)

print("\n===== RAW LLM OUTPUT =====\n")
print(node2)




===== RAW LLM OUTPUT =====

INTENT:
- Tìm kiếm nhà hàng quán nhậu hải sản tại Đà Nẵng

EXPANSION:
- Tìm kiếm thông tin về quán hải sản tại Đà Nẵng
- Tìm kiếm đánh giá và review về quán hải sản tại Đà Nẵng
- Tìm kiếm quán hải sản giá rẻ và chất lượng tại Đà Nẵng

CONTENT SIGNALS:
- price: giá rẻ, giá bình dân
- quality: tươi ngon, chất lượng
- format: quán ăn, nhà hàng
- location: Đà Nẵng, Mỹ Khê

SERP TYPE:
- informational
- navigational
- review

ENTITIES:
- Đà Nẵng
- Mỹ Khê

MARKET INSIGHT:
- Người dùng đang tìm kiếm thông tin về quán hải sản tại Đà Nẵng với mức giá hợp lý và chất lượng tốt.


In [3]:
import asyncio
from collections import Counter
from bs4 import BeautifulSoup
import nodriver as uc


# =========================
# CLEAN TEXT
# =========================
def clean_text(text: str):
    return " ".join(text.split()).strip()


# =========================
# DETECT SIMPLE STRUCTURE (GENERIC)
# =========================
def detect_structure(text: str):
    t = text.lower()

    score = {
        "listicle": t.count("1.") + t.count("2.") + t.count("top") + t.count("danh sách"),
        "review": t.count("trải nghiệm") + t.count("đánh giá"),
        "guide": t.count("hướng dẫn") + t.count("cách"),
        "informational": t.count("là gì") + t.count("khái niệm"),
    }

    return max(score, key=score.get)


# =========================
# EXTRACT CONTENT SIGNALS (NO DOMAIN KNOWLEDGE)
# =========================
def extract_signals(text: str):

    t = text.lower()

    signals = {
        "price": sum([t.count(x) for x in ["giá", "rẻ", "đắt", "chi phí"]]),
        "quality": sum([t.count(x) for x in ["ngon", "tốt", "chất lượng", "tươi"]]),
        "location": sum([t.count(x) for x in ["gần", "ở", "tại", "khu vực"]]),
        "action": sum([t.count(x) for x in ["đặt", "liên hệ", "xem thêm", "mua"]]),
    }

    return signals


# =========================
# CTA DETECTION (GENERIC)
# =========================
def detect_cta(text: str):

    t = text.lower()

    if "đặt" in t:
        return "transaction"
    if "liên hệ" in t:
        return "contact"
    if "xem thêm" in t:
        return "explore"
    if "mua" in t or "giá" in t:
        return "commercial"

    return "informational"


# =========================
# EXTRACT CORE CONTENT (TRIM NOISE)
# =========================
def extract_core(text: str):
    """
    Không lấy full text → chỉ lấy đoạn có signal cao
    """
    sentences = text.split(".")
    important = []

    keywords = ["giá", "ngon", "tốt", "review", "top", "địa chỉ", "đặt"]

    for s in sentences:
        if any(k in s.lower() for k in keywords):
            important.append(s.strip())

    return " | ".join(important[:5])


# =========================
# MAIN NODE 2 (GENERIC + PATTERN MINING)
# =========================
async def node3_competitor_parsing(node1):

    urls = node1.get("top_urls", [])

    browser = await uc.start(headless=True)

    raw_pages = []

    try:
        for item in urls:

            url = item.get("url")
            title = item.get("title", "")

            if not url:
                continue

            try:
                page = await browser.get(url)
                await asyncio.sleep(1.5)

                soup = BeautifulSoup(await page.get_content(), "html.parser")
                text = clean_text(soup.get_text(" "))

                raw_pages.append({
                    "url": url,
                    "title": title,
                    "structure": detect_structure(text),
                    "signals": extract_signals(text),
                    "cta": detect_cta(text),
                    "core": extract_core(text),
                    "length": len(text)
                })

            except Exception as e:
                raw_pages.append({
                    "url": url,
                    "title": title,
                    "error": str(e)
                })

    finally:
        browser.stop()


    # =========================
    # AGGREGATION LAYER (QUAN TRỌNG)
    # =========================

    structure_counter = Counter([x.get("structure") for x in raw_pages if "structure" in x])

    cta_counter = Counter([x.get("cta") for x in raw_pages if "cta" in x])

    avg_length = sum([x.get("length", 0) for x in raw_pages]) / max(len(raw_pages), 1)


    return {
        "pages": raw_pages,

        # 🔥 CORE INSIGHT (KHÔNG HARD CODE)
        "market_pattern": {
            "dominant_structure": structure_counter.most_common(1)[0][0] if structure_counter else "unknown",
            "cta_distribution": dict(cta_counter),
            "avg_content_length": int(avg_length)
        }
    }


# =========================
# RUN
# =========================
node3 = await node3_competitor_parsing(node1)

print("\n===== RAW LLM OUTPUT =====\n")
node3


===== NODE 2 OUTPUT =====



{'pages': [{'url': 'https://www.bachhoaxanh.com/kinh-nghiem-hay/10-quan-hai-san-gia-re-ngon-nuc-tieng-o-da-nang-ma-ban-co-the-de-dang-tim-thay-1406465',
   'title': '10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có ...',
   'structure': 'guide',
   'signals': {'price': 51, 'quality': 76, 'location': 62, 'action': 40},
   'cta': 'transaction',
   'core': '10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy DANH MỤC SẢN PHẨM Đăng nhập #Khuyến mãi sốc #Mẹo vặt cuộc sống #Món ngon mỗi ngày #Dọn dẹp nhà xinh #Mở tiệc tại gia #Chăm sóc cá nhân Tặng 10K Mua Nước Tăng Lực Redbull Từ 250 | 2KG KT: 30/06/2026 Lấy ngay #Kinh nghiệm hay > #Mẹo vặt cuộc sống > 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy Nếu bạn đang tìm kiếm các quán hải sản ngon tại Đà Nẵng | Thì hãy tìm hiểu 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng 

In [ ]:
import re
from collections import Counter


# =========================
# CLEAN TEXT
# =========================
def normalize(text):
    return re.sub(r"\s+", " ", text.lower()).strip()


# =========================
# BUILD TEXT CORPUS
# =========================
def build_corpus(node1, node2, node15):

    texts = []

    # Node 1 (SERP titles)
    for x in node1.get("top_urls", []):
        texts.append(x.get("title", ""))

    # Node 1.5 (intent + expansion)
    texts += node15.get("expansion_signals", [])
    texts += node15.get("keyword_cluster", [])

    # Node 2 (competitors)
    for c in node2.get("competitors", []):
        texts.append(c.get("title", ""))

    return [normalize(t) for t in texts if t]


# =========================
# MARKET POSITION (INSIGHT LAYER)
# =========================
def market_position(node2):

    types = [c.get("type") for c in node2.get("competitors", [])]
    angles = []

    for c in node2.get("competitors", []):
        angles += c.get("angle", [])

    type_freq = Counter(types)
    angle_freq = Counter(angles)

    dominant_type = type_freq.most_common(1)[0][0] if type_freq else "unknown"
    dominant_angles = [k for k, _ in angle_freq.most_common(3)]

    return {
        "dominant_content_type": dominant_type,
        "dominant_angles": dominant_angles
    }


# =========================
# GAP DETECTION (WHAT IS MISSING)
# =========================
def detect_gaps(node2):

    expected = {
        "storytelling",
        "experience",
        "local_insider",
        "comparison_depth",
        "pricing_breakdown",
        "vibe_based_review"
    }

    found = set()

    for c in node2.get("competitors", []):
        found.update(c.get("angle", []))

    missing = list(expected - found)

    return missing


# =========================
# WINNING STRATEGY GENERATION
# =========================
def winning_strategy(position, gaps, node15):

    intent = node15.get("intent_scores", [])

    top_intent = intent[0][0] if intent else "unknown"

    strategy = []

    # positioning logic
    if position["dominant_content_type"] == "listicle":
        strategy.append("Avoid pure SEO listicle format")
        strategy.append("Use experience-based narrative structure")

    if "experience" in gaps:
        strategy.append("Inject real local experience storytelling")

    if "storytelling" in gaps:
        strategy.append("Add human perspective instead of generic listing")

    if "vibe_based_review" in gaps:
        strategy.append("Differentiate via atmosphere / vibe description")

    # intent alignment
    strategy.append(f"Primary intent target: {top_intent}")

    return strategy


# =========================
# MAIN NODE 3 (FINAL OUTPUT)
# =========================
def node3_gap_engine(node1, node2, node15):

    corpus = build_corpus(node1, node2, node15)

    position = market_position(node2)

    gaps = detect_gaps(node2)

    strategy = winning_strategy(position, gaps, node15)

    return {
        "market_position": position,
        "content_gaps": gaps,
        "winning_strategy": strategy
    }


# =========================
# RUN
# =========================
node3 = node3_gap_engine(node1, node2, node15)

print("\n===== RAW LLM OUTPUT =====\n")
print(node3)

In [ ]:
# brand_voice_graph.py
from langgraph.graph import StateGraph, END

# Import 4 node
from scrape_competitors_node import scrape_competitors
from crawl_content_node import crawl_competitor_content
from extract_insights_node import extract_competitor_insights
from generate_brand_voice_node import generate_brand_voice

# Build graph
builder = StateGraph(dict)

builder.add_node("scrape_competitors", scrape_competitors)
builder.add_node("crawl_content", crawl_competitor_content)
builder.add_node("extract_insights", extract_competitor_insights)
builder.add_node("generate_brand_voice", generate_brand_voice)

# Flow: 1 → 2 → 3 → 4 → END
builder.set_entry_point("scrape_competitors")
builder.add_edge("scrape_competitors", "crawl_content")
builder.add_edge("crawl_content", "extract_insights")
builder.add_edge("extract_insights", "generate_brand_voice")
builder.add_edge("generate_brand_voice", END)

brand_voice_graph = builder.compile()

In [ ]:
# test_brand_voice.py
import asyncio

async def test():
    initial_state = {
        "industry": "nhà hàng hải sản",
        "location": "Đà Nẵng",
        "brand_info": {
            "name": "Mộc Quán",
            "industry": "Nhà hàng hải sản",
            "target": "Du khách và người địa phương Đà Nẵng",
            "usp": "Không gian mộc mạc phong cách Huế-Hội An, hải sản tươi sống"
        }
    }
    
    # Chạy từng node hoặc full graph
    result = await brand_voice_graph.ainvoke(initial_state)
    print(json.dumps(result["brand_voice"], indent=2, ensure_ascii=False))

asyncio.run(test())